# Case Study 15: US Traffic Accidents — Binary Severity Classification
## Aurora-GLM Showcase: Large-Scale Binomial GLM

---

## 1. Overview

This notebook demonstrates Aurora-GLM's **scalability** with a large dataset (~150K+ records). We predict whether an accident is **severe or minor** (binary outcome) using temporal, environmental, and road features with a Binomial GLM (logistic regression).

### Research Questions

1. Which factors predict severe accidents?
2. Do temporal patterns (rush hour, night) matter?
3. How does weather affect severity?
4. What GPU speedup is achieved at scale?

### Aurora-GLM Capabilities

1. Large-scale Binomial GLM (logit link)
2. GPU acceleration for big data
3. Odds ratio interpretation
4. Predictive metrics (accuracy, F1, AUC)

---

## 2. Setup and Data Generation

Synthetic data replicating the structure of the US Accidents dataset (Moosavi et al., 2019): **150,000 accidents** with temporal (hour, rush hour, night), environmental (temperature, humidity, visibility, precipitation, rain/fog) and road features (junction, traffic signal). Generated with a fixed seed (`np.random.seed(42)`) for reproducibility; the true severity model is documented in the generation cell.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import time
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from aurora.models.glm import fit_glm
from aurora.models.gam import fit_gam

try:
    import torch
    TORCH_AVAILABLE = True
    GPU_AVAILABLE = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'N/A'
except ImportError:
    TORCH_AVAILABLE = False
    GPU_AVAILABLE = False
    GPU_NAME = 'N/A'

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context('notebook', font_scale=1.1)
%config InlineBackend.figure_format = 'retina'
np.random.seed(42)

print("="*80)
print("ENVIRONMENT SETUP")
print("="*80)
print(f"PyTorch: {'Available' if TORCH_AVAILABLE else 'Not installed'}")
print(f"GPU: {'Available - ' + GPU_NAME if GPU_AVAILABLE else 'Not available'}")
print("="*80)

In [ ]:
# Generate large-scale synthetic accident data
# (Simulating US Accidents dataset structure)
print("="*80)
print("DATA GENERATION")
print("="*80)

np.random.seed(42)

# Generate 150,000 accidents for RAM efficiency
n_accidents = 150_000

# Temporal features
hour = np.random.choice(24, n_accidents, p=[
    0.02, 0.01, 0.01, 0.01, 0.02, 0.03, 0.05, 0.08,  # 0-7
    0.07, 0.05, 0.04, 0.04, 0.05, 0.05, 0.05, 0.06,  # 8-15
    0.07, 0.08, 0.07, 0.05, 0.03, 0.02, 0.02, 0.02   # 16-23
])
day_of_week = np.random.choice(7, n_accidents)  # 0=Mon
month = np.random.choice(12, n_accidents) + 1

# Weather conditions
temperature = np.random.normal(55, 20, n_accidents).clip(0, 110)  # Fahrenheit
humidity = np.random.beta(2, 2, n_accidents) * 100  # %
visibility = np.random.exponential(8, n_accidents).clip(0.1, 10)  # miles
precipitation = np.random.exponential(0.1, n_accidents).clip(0, 2)  # inches

# Road features
junction = np.random.binomial(1, 0.3, n_accidents)
crossing = np.random.binomial(1, 0.15, n_accidents)
traffic_signal = np.random.binomial(1, 0.2, n_accidents)
stop_sign = np.random.binomial(1, 0.1, n_accidents)

# Weather categories
weather_clear = np.random.binomial(1, 0.6, n_accidents)
weather_rain = np.random.binomial(1, 0.2, n_accidents) * (1 - weather_clear)
weather_fog = np.random.binomial(1, 0.1, n_accidents) * (1 - weather_clear) * (1 - weather_rain)

# Generate severity (binary: 0=minor, 1=severe)
# Rush hour effect
rush_hour = ((hour >= 7) & (hour <= 9)) | ((hour >= 16) & (hour <= 19))
night = (hour >= 20) | (hour <= 5)

# Calculate log-odds
logit_severe = (-2.0  # Baseline
    + 0.5 * rush_hour  # Rush hour increases severity
    + 0.8 * night  # Night increases severity
    - 0.02 * (temperature - 55)  # Extreme temps
    + 0.01 * humidity
    - 0.3 * visibility  # Low visibility increases severity
    + 1.0 * (precipitation > 0.5)  # Heavy rain
    + 0.3 * junction
    - 0.2 * traffic_signal  # Signals protective
    + 0.5 * weather_rain
    + 0.8 * weather_fog
)

prob_severe = 1 / (1 + np.exp(-logit_severe))
severe = np.random.binomial(1, prob_severe)

# Create DataFrame
df = pd.DataFrame({
    'hour': hour,
    'day_of_week': day_of_week,
    'month': month,
    'temperature': temperature,
    'humidity': humidity,
    'visibility': visibility,
    'precipitation': precipitation,
    'junction': junction,
    'crossing': crossing,
    'traffic_signal': traffic_signal,
    'stop_sign': stop_sign,
    'weather_clear': weather_clear,
    'weather_rain': weather_rain,
    'weather_fog': weather_fog,
    'rush_hour': rush_hour.astype(int),
    'night': night.astype(int),
    'severe': severe
})

# Standardize continuous
for col in ['temperature', 'humidity', 'visibility', 'precipitation']:
    df[f'{col}_std'] = (df[col] - df[col].mean()) / df[col].std()

print(f"\nDataset: {len(df):,} accidents")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nSeverity Rate: {df['severe'].mean()*100:.1f}% severe")
print(f"\nFeatures: {len(df.columns)}")
print("="*80)

## 3. Exploratory Data Analysis

In [ ]:
# EDA
print("="*80)
print("EXPLORATORY DATA ANALYSIS")
print("="*80)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Severity by hour
hourly_severity = df.groupby('hour')['severe'].mean() * 100
axes[0, 0].plot(hourly_severity.index, hourly_severity.values, 'o-', color='coral', linewidth=2)
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Severity Rate (%)')
axes[0, 0].set_title('Severity by Hour', fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# Severity by day
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily_severity = df.groupby('day_of_week')['severe'].mean() * 100
axes[0, 1].bar(range(7), daily_severity.values, color='steelblue', alpha=0.7)
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(days)
axes[0, 1].set_ylabel('Severity Rate (%)')
axes[0, 1].set_title('Severity by Day of Week', fontweight='bold')

# Visibility effect
vis_bins = pd.cut(df['visibility'], bins=10)
vis_severity = df.groupby(vis_bins, observed=True)['severe'].mean() * 100
centers = [i.mid for i in vis_severity.index]
axes[0, 2].plot(centers, vis_severity.values, 'o-', color='seagreen', linewidth=2)
axes[0, 2].set_xlabel('Visibility (miles)')
axes[0, 2].set_ylabel('Severity Rate (%)')
axes[0, 2].set_title('Severity by Visibility', fontweight='bold')

# Weather effect
weather_cats = ['Clear', 'Rain', 'Fog']
weather_severity = [
    df[df['weather_clear']==1]['severe'].mean() * 100,
    df[df['weather_rain']==1]['severe'].mean() * 100,
    df[df['weather_fog']==1]['severe'].mean() * 100
]
axes[1, 0].bar(weather_cats, weather_severity, color='purple', alpha=0.7)
axes[1, 0].set_ylabel('Severity Rate (%)')
axes[1, 0].set_title('Severity by Weather', fontweight='bold')

# Junction effect
junction_severity = df.groupby('junction')['severe'].mean() * 100
axes[1, 1].bar(['No Junction', 'Junction'], junction_severity.values, color='orange', alpha=0.7)
axes[1, 1].set_ylabel('Severity Rate (%)')
axes[1, 1].set_title('Severity by Junction', fontweight='bold')

# Traffic signal effect
signal_severity = df.groupby('traffic_signal')['severe'].mean() * 100
axes[1, 2].bar(['No Signal', 'Signal'], signal_severity.values, color='teal', alpha=0.7)
axes[1, 2].set_ylabel('Severity Rate (%)')
axes[1, 2].set_title('Severity by Traffic Signal', fontweight='bold')

plt.tight_layout()
plt.show()
print("="*80)

## 4. Mathematical Specification

**Family and link.** The outcome is binary ($y_i = 1$ if accident $i$ is severe, 0 otherwise), modeled as:

$$y_i \sim \text{Bernoulli}(\pi_i), \qquad \text{logit}(\pi_i) = \log\frac{\pi_i}{1-\pi_i} = \mathbf{x}_i^T \boldsymbol{\beta}$$

**Likelihood.** The Bernoulli (Binomial with $n_i = 1$) log-likelihood is:

$$\ell(\boldsymbol{\beta}) = \sum_{i=1}^{n} \left[ y_i \, \mathbf{x}_i^T\boldsymbol{\beta} - \log\left(1 + e^{\mathbf{x}_i^T\boldsymbol{\beta}}ight) \right]$$

maximized by IRLS (Fisher scoring) in `fit_glm`.

**Interpretation.** $e^{\beta_j}$ is the **odds ratio** for a one-unit increase in $x_j$: values above 1 raise the odds of a severe accident, below 1 lower them.

**Evaluation.** Beyond fit statistics (AIC), predictive performance is assessed with accuracy, sensitivity/specificity, F1, and the area under the ROC curve (AUC, computed with `np.trapezoid`).


## 5. Model Fitting

In [ ]:
# Model fitting
print("="*80)
print("MODEL FITTING: BINOMIAL GLM")
print("="*80)

# Design matrix
X = np.column_stack([
    np.ones(len(df)),
    df['rush_hour'].values,
    df['night'].values,
    df['temperature_std'].values,
    df['humidity_std'].values,
    df['visibility_std'].values,
    df['precipitation_std'].values,
    df['junction'].values,
    df['traffic_signal'].values,
    df['weather_rain'].values,
    df['weather_fog'].values
])
y = df['severe'].values

predictor_names = ['Intercept', 'Rush_Hour', 'Night', 'Temperature', 'Humidity',
                   'Visibility', 'Precipitation', 'Junction', 'Traffic_Signal',
                   'Rain', 'Fog']

print(f"Design Matrix: {X.shape}")
print(f"This is a LARGE dataset - GPU will show significant speedup\n")

# NumPy backend
print("Fitting with NumPy backend...")
start_time = time.time()
# fit_intercept=False: X already includes an explicit intercept column
result_numpy = fit_glm(X=X, y=y, family='binomial', link='logit', fit_intercept=False)
time_numpy = time.time() - start_time

print(f"\nConverged: {result_numpy.converged_}")
print(f"Iterations: {result_numpy.n_iter_}")
print(f"Time: {time_numpy:.3f}s")
print(f"AIC: {result_numpy.aic_:.2f}")
print("="*80)

## 6. Residual Diagnostics

With a binary outcome and N=150K, individual residuals are uninformative; we use **binned deviance residuals** (mean deviance residual within groups of similar fitted probability) and a **calibration plot** (observed severity rate vs mean predicted probability per bin).

In [ ]:
print("="*80)
print("RESIDUAL DIAGNOSTICS")
print("="*80)

# Fitted probabilities (fit_intercept=False: X carries its own intercept column,
# so X @ coef_ and predict() coincide)
prob = np.asarray(result_numpy.predict(X))
prob_clip = np.clip(prob, 1e-10, 1 - 1e-10)

# Deviance residuals
d = np.sign(y - prob_clip) * np.sqrt(-2 * (y * np.log(prob_clip) + (1 - y) * np.log(1 - prob_clip)))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: binned deviance residuals vs fitted
bins = pd.qcut(prob, q=20, duplicates='drop')
binned = pd.DataFrame({'prob': prob, 'd': d}).groupby(bins, observed=True).agg(
    mean_prob=('prob', 'mean'), mean_resid=('d', 'mean'))
axes[0].plot(binned['mean_prob'], binned['mean_resid'], 'o-', color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Fitted probability (binned)')
axes[0].set_ylabel('Mean deviance residual')
axes[0].set_title('Binned Deviance Residuals vs Fitted', fontweight='bold')
axes[0].grid(alpha=0.3)

# Panel 2: distribution of deviance residuals
axes[1].hist(d, bins=100, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Deviance residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Deviance Residuals', fontweight='bold')
axes[1].grid(alpha=0.3)

# Panel 3: calibration plot
cal_bins = pd.qcut(prob, q=15, duplicates='drop')
cal = pd.DataFrame({'prob': prob, 'y': y}).groupby(cal_bins, observed=True).agg(
    mean_pred=('prob', 'mean'), obs_rate=('y', 'mean'), n=('y', 'size'))
axes[2].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect calibration')
axes[2].scatter(cal['mean_pred'], cal['obs_rate'], s=np.sqrt(cal['n']), alpha=0.7, color='steelblue')
axes[2].plot(cal['mean_pred'], cal['obs_rate'], 'b-', alpha=0.5)
axes[2].set_xlabel('Mean predicted probability')
axes[2].set_ylabel('Observed severity rate')
axes[2].set_title('Calibration Plot', fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

cal_err = np.abs(cal['obs_rate'] - cal['mean_pred']).mean()
print(f"\nMean deviance residual: {d.mean():.4f}")
print(f"Max |binned residual|:  {np.abs(binned['mean_resid']).max():.4f} across {len(binned)} bins")
print(f"Mean calibration error: {cal_err:.4f} across {len(cal)} bins")
print("   ✓ Well calibrated" if cal_err < 0.05 else "   ⚠ Calibration error above 5 points")
print("\n" + "="*80)

## 7. Interpretation and Predictive Performance

Odds ratios on the response scale, then classification metrics (threshold 0.5) and ROC/AUC.

In [ ]:
# Odds ratios
print("="*80)
print("ODDS RATIOS")
print("="*80)

print("\nOdds Ratios for Severe Accident:")
print("-" * 60)

for name, coef in zip(predictor_names, result_numpy.coef_):
    if name == 'Intercept':
        continue
    or_val = np.exp(coef)
    pct = (or_val - 1) * 100
    direction = 'increases' if or_val > 1 else 'decreases'
    print(f"{name:20s}: OR={or_val:.3f} ({pct:+.1f}% {direction} odds)")

print("="*80)

In [ ]:
# Predictive performance
print("="*80)
print("PREDICTIVE PERFORMANCE")
print("="*80)

# Predictions
eta = X @ result_numpy.coef_
prob = 1 / (1 + np.exp(-eta))
pred = (prob > 0.5).astype(int)

# Metrics
accuracy = (pred == y).mean()
tp = ((pred == 1) & (y == 1)).sum()
tn = ((pred == 0) & (y == 0)).sum()
fp = ((pred == 1) & (y == 0)).sum()
fn = ((pred == 0) & (y == 1)).sum()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1 = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

# AUC
thresholds = np.linspace(0, 1, 100)
tpr_list, fpr_list = [], []
for t in thresholds:
    pred_t = (prob > t).astype(int)
    tpr = ((pred_t == 1) & (y == 1)).sum() / max(1, (y == 1).sum())
    fpr = ((pred_t == 1) & (y == 0)).sum() / max(1, (y == 0).sum())
    tpr_list.append(tpr)
    fpr_list.append(fpr)
auc = -np.trapezoid(tpr_list, fpr_list)

print(f"\nClassification Metrics:")
print(f"   Accuracy: {accuracy:.3f}")
print(f"   Sensitivity: {sensitivity:.3f}")
print(f"   Specificity: {specificity:.3f}")
print(f"   Precision: {precision:.3f}")
print(f"   F1 Score: {f1:.3f}")
print(f"   AUC: {auc:.3f}")

# ROC curve
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_list, tpr_list, 'b-', linewidth=2, label=f'ROC (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print("="*80)

## 8. Multi-Backend Performance

Same fit through the available backends. Only installed backends are benchmarked (PyTorch/JAX guarded); at N=150K the GPU path is where this section would show speedups on equipped machines.

In [ ]:
# CRITICAL: Multi-backend benchmark for large dataset
print("="*80)
print("MULTI-BACKEND PERFORMANCE BENCHMARK")
print("="*80)
print(f"\nDataset size: {len(df):,} records")
print("This benchmark demonstrates GPU advantage for large-scale GLM\n")

benchmark_results = [
    {'Backend': 'NumPy (CPU)', 'Time (s)': f'{time_numpy:.3f}', 'Speedup': '1.00x'}
]

# PyTorch CPU
if TORCH_AVAILABLE:
    print("Benchmarking PyTorch CPU...")
    start = time.time()
    result_cpu = fit_glm(X=X, y=y, family='binomial', link='logit',
                        backend='torch', device='cpu', fit_intercept=False)
    time_cpu = time.time() - start
    speedup_cpu = time_numpy / time_cpu
    benchmark_results.append({
        'Backend': 'PyTorch (CPU)',
        'Time (s)': f'{time_cpu:.3f}',
        'Speedup': f'{speedup_cpu:.2f}x'
    })
    print(f"   Time: {time_cpu:.3f}s")

# PyTorch GPU - THE KEY BENCHMARK
if GPU_AVAILABLE:
    print(f"\nBenchmarking PyTorch GPU ({GPU_NAME})...")
    
    # Warm-up
    _ = fit_glm(X=X[:5000], y=y[:5000], family='binomial', link='logit',
               backend='torch', device='cuda', fit_intercept=False)
    
    start = time.time()
    result_gpu = fit_glm(X=X, y=y, family='binomial', link='logit',
                        backend='torch', device='cuda', fit_intercept=False)
    time_gpu = time.time() - start
    speedup_gpu = time_numpy / time_gpu
    
    benchmark_results.append({
        'Backend': f'PyTorch (GPU)',
        'Time (s)': f'{time_gpu:.3f}',
        'Speedup': f'{speedup_gpu:.2f}x'
    })
    
    print(f"   Time: {time_gpu:.3f}s")
    print(f"   GPU SPEEDUP: {speedup_gpu:.2f}x faster than NumPy!")
    
    # Verify results match - convert GPU coefficients to NumPy
    coef_gpu_np = result_gpu.coef_.cpu().numpy() if torch.is_tensor(result_gpu.coef_) else result_gpu.coef_
    coef_diff = np.max(np.abs(result_numpy.coef_ - coef_gpu_np))
    print(f"   Max coefficient difference: {coef_diff:.6f} (numerical precision)")

print("\n" + "="*80)
print("BENCHMARK SUMMARY")
print("="*80)
print(pd.DataFrame(benchmark_results).to_string(index=False))

if GPU_AVAILABLE:
    print(f"\n*** RTX 5070 Ti achieves {speedup_gpu:.1f}x speedup on {len(df):,} records ***")

print("="*80)

## 9. Conclusions

### Key Findings (by Research Question)

**RQ1 — Which factors predict severe accidents?**
- **Night** (OR=2.28) and **fog** (OR=2.28) are the strongest risk factors
- **Rain** (OR=1.72) and **rush hour** (OR=1.70) also raise severity odds substantially
- **Visibility** is strongly protective in this parameterization (OR=0.34 per SD — higher visibility, lower severity)
- **Traffic signals** are protective (OR=0.80); **junctions** increase risk (OR=1.35)

**RQ2 — Do temporal patterns matter?** Yes: night more than doubles severity odds, and rush hour raises them ~70% (congestion paradox: more but less severe accidents is NOT what this synthetic DGP encodes — rush hour increases severity here).

**RQ3 — How does weather affect severity?** Fog and rain increase severity odds; higher temperature slightly decreases them (OR=0.67 per SD).

**RQ4 — What GPU speedup is achieved at scale?** None measured here: PyTorch/JAX are not installed in this environment, so only NumPy ran (0.165 s for 150K × 11 — already fast). The `backend='torch'` path exists and is where GPU speedups would appear on equipped machines.

### Predictive Performance

- Accuracy 0.898, AUC 0.790 — but **sensitivity is only 0.030 at threshold 0.5**: the severe class is rare, so the model almost never predicts it at the default threshold. Accuracy is therefore misleading here; threshold tuning (or class-weighted fitting) is required for a usable severity alarm.

### Limitations

- **Synthetic data**: effect sizes are built into the DGP; real accident data would show weaker, noisier effects and unmeasured confounding (speed, alcohol, vehicle type are absent).
- **Class imbalance**: severe accidents are rare; the default 0.5 threshold yields sensitivity ≈ 0.03. A decision-theoretic threshold (cost of missing a severe accident vs false alarms) should be chosen instead.
- **Linearity on the logit scale**: no interactions (e.g., night × fog) or non-linear weather effects; a GAM or interaction terms could capture these.
- **In-sample metrics**: no train/test split; AUC is optimistic.
- **Correlation between predictors**: e.g., rain and precipitation overlap; odds ratios are conditional on the full set.

### References

- McCullagh, P., & Nelder, J. A. (1989). *Generalized Linear Models* (2nd ed.). Chapman & Hall/CRC.
- Hosmer, D. W., Lemeshow, S., & Sturdivant, R. X. (2013). *Applied Logistic Regression* (3rd ed.). Wiley.
- Moosavi, S., Samavatian, M. H., Parthasarathy, S., & Ramnath, R. (2019). "A Countrywide Traffic Accident Dataset." *arXiv:1906.05409* (structure replicated synthetically here).

---
**Dataset**: Simulated US Traffic Accidents (N=150,000, seed 42)
**Model**: Binomial GLM (logit link), 10 predictors
